In [ ]:
# Data: ../csv/Fig2_B_Example_replay/  (the original loading code is kept below each data line, as comments)
import os, glob, pickle, re, sys   # standard imports
from pathlib import Path
sys.path.insert(0, str(Path('..') / '..' / '..' / 'methods'))   # the criticality library
import pandas as pd
import numpy as np
CSV = Path('..') / 'csv' / 'Fig2_B_Example_replay'
OUT = Path('..') / 'figure_output'
OUT.mkdir(exist_ok=True)


def read_csv(name, **kw):
    """A table of this notebook (round_trip: the floats exactly as written)."""
    return pd.read_csv(CSV / name, float_precision='round_trip', **kw)

In [ ]:
import matplotlib.pyplot as plt
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from neuro_py.plotting.figure_helpers import set_size
from neuro_py.tuning import maps
from skimage import measure
import os
import pandas as pd
import scipy.io as sio
import logging

logging.getLogger().setLevel(logging.ERROR)
basepath =  r"W:\data\PPP\PPP21\day4"
basename=os.path.basename(basepath)# Define the new folder path
#create working folder
new_folder_path = os.path.join(basepath, "complex_system")
brain_region="CA1"
animal="mouse"   #or Mouse
cell_type='all'
path_parts = basepath.split("\\")  # Split the path using backslash
animal_name = path_parts[-2]  # Second-to-last part is the animal name
day = path_parts[-1]  # Last part is the day
# ---- data (../csv) ----
save_dir = str(OUT)
nrem_epochs = nel.EpochArray(read_csv('states_NREM.csv')[['start', 'stop']].to_numpy())
awake_epochs = nel.EpochArray(read_csv('states_WAKE.csv')[['start', 'stop']].to_numpy())
rem_epochs = nel.EpochArray(read_csv('states_REM.csv')[['start', 'stop']].to_numpy())
epoch_df = read_csv('epochs.csv')
sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
spike_times = None
# ---- the same from the recordings, with analysis_scripts/: ----
# save_dir = r"D:\Documents\GitHub\R_Project\PPP\codebook\Criticality_analysis\complex_system\manifold"
# if not os.path.exists(save_dir):
#     os.makedirs(save_dir)
# # Create the folder if it doesn't exist
# if not os.path.exists(new_folder_path):
#     os.makedirs(new_folder_path)
#     print(f"Folder created: {new_folder_path}")
# else:
#     print(f"Folder already exists: {new_folder_path}")
# # load in spike data from hpc pyramidal cells
# st, cm = loading.load_spikes(
#     basepath, brainRegion=brain_region                          # putativeCellType="Pyr"
# )
# # load theta epochs
# state_dict = loading.load_SleepState_states(basepath)
# nrem_epochs = nel.EpochArray(
#     state_dict["NREMstate"],
# )
# awake_epochs = nel.EpochArray(
#     state_dict["WAKEstate"],
# )
# # theta_epochs = nel.EpochArray(
# #     state_dict["THETA"],
# # )
# rem_epochs=nel.EpochArray(state_dict["REMstate"],)
# try:
#     # Try default loader
#     epoch_df = loading.load_epoch(basepath)
# except Exception as e:
#     print(f"load_epoch failed with error: {e}")
#     print("Falling back to manual .mat file load...")
# 
#     # Load the .mat file
#     mat = sio.loadmat(os.path.join(basepath, "DayNight25.MergePoints.events.mat"), squeeze_me=True)
# 
#     # Extract the 'MergePoints' structure
#     merge_points = mat['MergePoints']
# 
#     # Extract the timestamps array: shape (N, 2)
#     timestamps = merge_points['timestamps']
#     # Unwrap the array inside
#     timestamps = np.array(timestamps.tolist())
# 
#     # shape (N, 2)
#     epoch_df = pd.DataFrame(timestamps, columns=["startTime", "stopTime"])
# 
# sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
# E_spikes = st.time[(cm.putativeCellType == 'Pyramidal Cell')]
# I_spikes= st.time[(cm.putativeCellType.str.contains("Interneuron"))]
# spike_times=st.time
# n_cell=len(spike_times)
# print(f"number of cells: {n_cell}")

In [ ]:
epoch_df

In [ ]:
# ---- data (../csv) ----
pass  # ripple and barrage events are not used by the panels of this notebook
# ---- the same from the recordings, with analysis_scripts/: ----
# import numpy as np
# import scipy.io as sio      
# import scipy
# # Load ripple events
# ripple_events = loading.load_ripples_events(basepath=basepath)
# ripple_epochs = nel.EpochArray(np.array([ripple_events.start, ripple_events.stop]).T)
# ripple_peaks=ripple_events.peaks
#     # Load barrage events
# ripple_epochs=nrem_epochs&ripple_epochs
# try:
#     # Load barrage events
#     #brpath=r"W:\data\PPP\CA2\PPP7\day8"
#     barrage_events = loading.load_barrage_events(basepath=basepath)  
#     # Create barrage epochs
#     barrage_epochs = nel.EpochArray(np.array([barrage_events.start, barrage_events.stop]).T)
#     # Find barrages in NREM epochs
#     barrages_in_nrem = barrage_epochs & nrem_epochs
#     mat_path = os.path.join(basepath, f'{basename}.HSEn2.events.mat')
# 
#     data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
#     barrage = data['HSEn2']
#     barrage_timestamps= barrage.timestamps 
#     barrage_epochs=nrem_epochs&barrage_epochs
# except Exception as e:
#     # General exception for unexpected errors
#     print(f"No barrage file: {e}")
#     barrage_epochs=[]
#     # Extract start times

In [ ]:
epoch_df

In [ ]:
# ---- data (../csv) ----
import numpy as np
import matplotlib.pyplot as plt
# every significant replay event (left then right)
_ev = read_csv('replay_events_significant.csv')
left_replay_matrix, right_replay_matrix, left_intervals, right_intervals = [], [], [], []
left_replay_goodfit, right_replay_goodfit, left_replay_slope, right_replay_slope = [], [], [], []
for _r in _ev.itertuples():
    _L = _r.side == 'left'
    (left_replay_matrix if _L else right_replay_matrix).append(None)
    (left_intervals if _L else right_intervals).append(np.array([_r.start, _r.stop]))
    (left_replay_goodfit if _L else right_replay_goodfit).append(_r.goodfit)
    (left_replay_slope if _L else right_replay_slope).append(_r.slope)
# the decoded posterior of the plotted event (1731)
_m = read_csv('replay_matrix_event1731.csv').to_numpy(float)
if 1731 < len(left_replay_matrix): left_replay_matrix[1731] = _m
else: right_replay_matrix[1731 - len(left_replay_matrix)] = _m
# ---- the same from the recordings, with analysis_scripts/: ----
# import scipy.io
# import numpy as np
# from scipy.ndimage import zoom
# from scipy.interpolate import interp1d
# import umap
# import matplotlib.pyplot as plt
# 
# # Load data
# mat_path = f'D:/Documents/GitHub/R_Project/PPP/codebook/Criticality_analysis/replay_results/{animal_name}_{day}_replay_sequences.mat'
# data = scipy.io.loadmat(mat_path)
# 
# # Define epochs and templates
# epochs = ['epoch1','epoch3','epoch5']
# template = 'template04'  # only use template02
# bin_size = 0.01  # 10 ms
# target_bins = 30  # fixed number of bins for all events
# 
# # Assume these are already loaded:
# # E_spikes: list of spike times (one per unit)
# # E_CellID: list of unit IDs corresponding to E_spikes
# 
# binned_firing_rates = []
# avg_pos = []
# epoch_labels = []
# left_replay = []
# right_replay = []
# left_replay = []
# left_times  = []
# 
# right_replay = []
# right_times  = []
# left_intervals=[]
# right_intervals=[]
# left_replay_goodfit=[]
# right_replay_goodfit=[]
# left_replay_slope=[]
# right_replay_slope=[]
# left_all_bestseq = []
# right_all_bestseq = []
# left_replay_matrix=[]
# right_replay_matrix=[]
# for e in epochs:
#     key_event = f'{e}_{template}_eventtimes'
#     key_seq   = f'{e}_{template}'
#     key_pval  = f'{e}_{template}_pvalue'
#     key_Goodfit  = f'{e}_{template}_Goodfit'
#     key_slope  = f'{e}_{template}_slope'
#     key_best_seq  = f'{e}_{template}_all_best_seq'
#     if key_event not in data or key_seq not in data:
#         continue
# 
#     events = data[key_event].squeeze()
#     seqs   = data[key_seq][0]
#     pvals  = data[key_pval].squeeze()
#     Goodfit  = data[key_Goodfit].squeeze()
#     slope  = data[key_slope].squeeze()
#     best_seq   = data[key_best_seq].squeeze()
#     if events.ndim == 1:
#         events = np.expand_dims(events, axis=0)
# 
#     for i, event in enumerate(events):
#         if i >= len(seqs):
#             continue
# 
#         pval = np.squeeze(pvals[i])   # [left_pval, right_pval]
#         s = np.squeeze(seqs[i])       # decoded sequence, shape (T,)
#         all_bestseq = np.squeeze(best_seq[i])       # decoded sequence, shape (T,)
# 
#         s_half = s.shape[1] // 2
#         event_time = np.mean(event)  # use midpoint time
# 
#         # Left significant
#         if pval[0] < 0.05:
#             left_replay.append(np.argmax(s[:,:s_half], axis=0) )
#             left_times.append(event_time)
#             left_intervals.append(event)
#             left_replay_goodfit.append(Goodfit[i,0])
#             left_replay_slope.append(slope[i][0, 0])
#             left_all_bestseq.append(all_bestseq[:s_half])
#             left_replay_matrix.append(s[:, :s_half])
#         # Right significant
#         if pval[1] < 0.05:
#             right_replay.append(np.argmax(s[:,s_half:], axis=0))
#             right_times.append(event_time)
#             right_intervals.append(event)
#             right_replay_goodfit.append(Goodfit[i,1])
#             right_replay_slope.append(slope[i][0, 1])
#             right_all_bestseq.append(all_bestseq[s_half:])
#             right_replay_matrix.append(s[:, s_half:]) 

In [ ]:
# ---- data (../csv) ----
_ns = read_csv('replay_events_nonsignificant.csv')
non_sig_matrices = [{'left': None, 'right': None, 'interval': np.array([r.start, r.stop]),
                     'pval': np.array([r.pval_left, r.pval_right])} for r in _ns.itertuples()]
non_sig_matrices[934]['left'] = read_csv('nonreplay_matrix_event934_left.csv').to_numpy(float)
non_sig_intervals = [d['interval'] for d in non_sig_matrices]
# ---- the same from the recordings, with analysis_scripts/: ----
# # Collect non-significant replay events
# non_sig_intervals = []
# non_sig_matrices = []
# 
# for e in epochs:
#     key_event = f'{e}_{template}_eventtimes'
#     key_seq   = f'{e}_{template}'
#     key_pval  = f'{e}_{template}_pvalue'
#     
#     if key_event not in data or key_seq not in data:
#         continue
#     
#     events = data[key_event].squeeze()
#     seqs   = data[key_seq][0]
#     pvals  = data[key_pval].squeeze()
#     
#     if events.ndim == 1:
#         events = np.expand_dims(events, axis=0)
#     
#     for i, event in enumerate(events):
#         if i >= len(seqs):
#             continue
#         
#         pval = np.squeeze(pvals[i])
#         s = np.squeeze(seqs[i])
#         s_half = s.shape[1] // 2
#         
#         # Check if NEITHER left nor right is significant
#         if pval[0] >= 0.05 and pval[1] >= 0.05:
#             non_sig_intervals.append(event)
#             # Store both halves, pick one randomly later
#             non_sig_matrices.append({
#                 'left': s[:, :s_half],
#                 'right': s[:, s_half:],
#                 'interval': event,
#                 'pval': pval
#             })
# 
# print(f"Found {len(non_sig_intervals)} non-significant replay events")
# print(f"Found {len(left_replay_matrix)} left significant replays")
# print(f"Found {len(right_replay_matrix)} right significant replays")

In [ ]:
chan = 255  # ripple channel
# ---- data (../csv) ----
epoch_df = read_csv('epochs.csv')
_meta = read_csv('lfp_meta.csv')
nChannels, fs, fs_dat = int(_meta.nChannels[0]), _meta.fs[0], _meta.fs_dat[0]
# LFP: only channel `chan`, only +-0.3 s around the two plotted events (the full .dat is GBs)
_lfp = read_csv('lfp_ch255_around_plotted_events.csv')
ts = _lfp['ts'].to_numpy(float)
lfp = np.zeros((len(ts), nChannels)); lfp[:, chan] = _lfp['lfp'].to_numpy(float)
# ---- the same from the recordings, with analysis_scripts/: ----
# epoch_df = loading.load_epoch(basepath) 
# ripples = loading.load_ripples_events(basepath, return_epoch_array=True) 
# nChannels, fs, fs_dat, shank_to_channel = loading.loadXML(basepath) 
# lfp,ts = loading.loadLFP(basepath,n_channels=nChannels,frequency=fs_dat,ext='dat')

In [ ]:
# ---- data (../csv) ----
pass  # (LFP size printout)
# ---- the same from the recordings, with analysis_scripts/: ----
# # Check LFP dimensions
# print(f"LFP shape: {lfp.shape}")
# print(f"Timestamp shape: {ts.shape}")
# print(f"Number of channels: {nChannels}")
# print(f"Requested channel: {chan}")
# print(f"Valid channel range: 0-{lfp.shape[0]-1}")

In [ ]:
# Example significant replay event
# Combine left and right significant replays
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [
    'Helvetica Neue', 'Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans'
]

# Mathtext to match
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Helvetica Neue'   # or 'Helvetica'
mpl.rcParams['mathtext.it'] = 'Helvetica Neue:italic'
mpl.rcParams['mathtext.bf'] = 'Helvetica Neue:bold'

# (optional) sizes + minus sign
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False

# Global sizes (tweak to taste)
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
all_sig_replays = []
for i, matrix in enumerate(left_replay_matrix):
    all_sig_replays.append({
        'matrix': matrix,
        'interval': left_intervals[i],
        'side': 'left',
        'goodfit': left_replay_goodfit[i],
        'slope': left_replay_slope[i]
    })
import numpy as np

def fit_replay_ridge_line(P, min_col_mass_frac=0.2):
    """
    Fit a line to the replay 'ridge' in a replay probability matrix P (pos x time).
    Returns: dict with x_fit, y_fit, m, b, r2, x_used, y_used
    """
    P = np.asarray(P)
    n_pos, n_t = P.shape

    # Ridge: for each time bin, take position of max probability
    y_ridge = np.argmax(P, axis=0).astype(float)  # pos index
    x = np.arange(n_t).astype(float)              # time index

    # Gate: keep time bins where column mass is nontrivial
    col_mass = P.sum(axis=0)
    thr = min_col_mass_frac * np.nanmax(col_mass) if np.nanmax(col_mass) > 0 else 0
    keep = col_mass >= thr

    x_used = x[keep]
    y_used = y_ridge[keep]

    if len(x_used) < 2:
        return {"m": np.nan, "b": np.nan, "r2": np.nan,
                "x_fit": None, "y_fit": None, "x_used": x_used, "y_used": y_used}

    # Fit line: y = m x + b
    m, b = np.polyfit(x_used, y_used, 1)

    # R^2 on used points
    y_hat = m * x_used + b
    ss_res = np.sum((y_used - y_hat) ** 2)
    ss_tot = np.sum((y_used - np.mean(y_used)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    x_fit = np.array([x_used.min(), x_used.max()])
    y_fit = m * x_fit + b

    return {"m": m, "b": b, "r2": r2,
            "x_fit": x_fit, "y_fit": y_fit, "x_used": x_used, "y_used": y_used}

for i, matrix in enumerate(right_replay_matrix):
    all_sig_replays.append({
        'matrix': matrix,
        'interval': right_intervals[i],
        'side': 'right',
        'goodfit': right_replay_goodfit[i],
        'slope': right_replay_slope[i]
    })

if len(all_sig_replays) > 0:
    random_idx = 1731
    random_event = all_sig_replays[random_idx]
    
    replay_matrix = random_event['matrix']
    interval = random_event['interval']
    side = random_event['side']
    goodfit = random_event['goodfit']
    slope = random_event['slope']
    
    # Get LFP around the interval
    center_time = np.mean(interval)
    lfp_window = 0.1  # 500ms window around event
    start_time = center_time - lfp_window
    end_time = center_time + lfp_window
    
    # Use valid channel index
    lfp_chan = min(chan, lfp.shape[1] - 1)
    
    # Find LFP indices
    start_idx = np.searchsorted(ts, start_time)
    end_idx = np.searchsorted(ts, end_time)
    
    # Keep indices within bounds
    start_idx = max(0, start_idx)
    end_idx = min(len(ts), end_idx)
    
    # LFP is (n_samples, n_channels), so index [start:end, channel]
    lfp_segment = lfp[start_idx:end_idx, lfp_chan]
    time_segment = ts[start_idx:end_idx]
    
    # Create figure with 2 subplots
    fig, axes = plt.subplots(2, 1, figsize=(5, 8), gridspec_kw={'height_ratios': [2, 1]})
    
    # Plot replay matrix

    ax1 = axes[0]
    dt_bin = 0.01  # 10 ms in seconds
    n_pos, n_t = replay_matrix.shape

    # imshow with bin EDGES: [0, n_t*dt]
    im = ax1.imshow(
        replay_matrix,
        aspect='auto',
        cmap='hot',
        origin='lower',
        extent=[0, n_t * dt_bin, 0, n_pos],
        vmin=0,          # lower color limit
        vmax=0.3         # upper color limit (example)
    )

    # --- Fit + overlay ridge line (convert bins -> seconds) ---
    fit = fit_replay_ridge_line(replay_matrix, min_col_mass_frac=0.2)

    if np.isfinite(fit["m"]):
        # convert BIN INDEX -> seconds at BIN CENTERS
        x_fit_sec  = (fit["x_fit"]  + 0.5) * dt_bin
        x_used_sec = (fit["x_used"] + 0.5) * dt_bin

        ax1.plot(x_fit_sec, fit["y_fit"], linewidth=2, color='cyan')


    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Position bins")
    plt.colorbar(im, ax=ax1, label="Probability")



    
    # Plot LFP
    ax2 = axes[1]
    if len(lfp_segment) > 0:
        ax2.plot(time_segment - center_time, lfp_segment, 'k', linewidth=0.5)
        ax2.axvspan(interval[0] - center_time, interval[1] - center_time, 
                    alpha=0.3, color='green', label='Replay interval')
        ax2.set_xlabel('Time relative to event center (s)')
        ax2.set_ylabel('LFP (µV)')
        ax2.set_title(f'LFP on channel {lfp_chan}')
        for spine in ["top", "right", "bottom", "left"]:
            ax2.spines[spine].set_visible(False)
        ax2.set_xticks([])
        ax2.set_yticks([])
        ax2.tick_params(
            bottom=False, left=False,
            labelbottom=False, labelleft=False
        )


    
    plt.tight_layout()
    save_dir=r"../figure_output"
    save_path = os.path.join(save_dir, f"fig2_B_Example_{animal_name}_{day}_replay_event_{random_idx}_{side}.svg")
    plt.savefig(save_path, dpi=300)
    plt.show()
    
    event_duration = interval[1] - interval[0]
else:
    pass

In [ ]:
# Example significant replay event
import matplotlib as mpl
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [
    'Helvetica Neue', 'Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans'
]

# Mathtext to match
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Helvetica Neue'   # or 'Helvetica'
mpl.rcParams['mathtext.it'] = 'Helvetica Neue:italic'
mpl.rcParams['mathtext.bf'] = 'Helvetica Neue:bold'

# (optional) sizes + minus sign
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False

# Global sizes (tweak to taste)
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
def fit_replay_ridge_line(P, min_col_mass_frac=0.2):
    """
    Fit a line to the replay 'ridge' in a replay probability matrix P (pos x time).
    Returns: dict with x_fit, y_fit, m, b, r2, x_used, y_used
    """
    P = np.asarray(P)
    n_pos, n_t = P.shape

    # Ridge: for each time bin, take position of max probability
    y_ridge = np.argmax(P, axis=0).astype(float)  # pos index
    x = np.arange(n_t).astype(float)              # time index

    # Gate: keep time bins where column mass is nontrivial
    col_mass = P.sum(axis=0)
    thr = min_col_mass_frac * np.nanmax(col_mass) if np.nanmax(col_mass) > 0 else 0
    keep = col_mass >= thr

    x_used = x[keep]
    y_used = y_ridge[keep]

    if len(x_used) < 2:
        return {"m": np.nan, "b": np.nan, "r2": np.nan,
                "x_fit": None, "y_fit": None, "x_used": x_used, "y_used": y_used}

    # Fit line: y = m x + b
    m, b = np.polyfit(x_used, y_used, 1)

    # R^2 on used points
    y_hat = m * x_used + b
    ss_res = np.sum((y_used - y_hat) ** 2)
    ss_tot = np.sum((y_used - np.mean(y_used)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    x_fit = np.array([x_used.min(), x_used.max()])
    y_fit = m * x_fit + b

    return {"m": m, "b": b, "r2": r2,
            "x_fit": x_fit, "y_fit": y_fit, "x_used": x_used, "y_used": y_used}



if len(non_sig_matrices) > 0:
    random_idx = 934
    random_event = non_sig_matrices[random_idx]
    
    # Randomly choose left or right half
    side = np.random.choice(['left', 'left'])
    replay_matrix = random_event[side]
    interval = random_event['interval']
    pval = random_event['pval']
    
    
    # Get LFP around the interval
    center_time = np.mean(interval)
    lfp_window = 0.1  # 500ms window around event
    start_time = center_time - lfp_window
    end_time = center_time + lfp_window
    
    # Use valid channel index
    lfp_chan = min(chan, lfp.shape[1] - 1)
    
    # Find LFP indices
    start_idx = np.searchsorted(ts, start_time)
    end_idx = np.searchsorted(ts, end_time)
    
    # Keep indices within bounds
    start_idx = max(0, start_idx)
    end_idx = min(len(ts), end_idx)
    
    # LFP is (n_samples, n_channels), so index [start:end, channel]
    lfp_segment = lfp[start_idx:end_idx, lfp_chan]
    time_segment = ts[start_idx:end_idx]
    
    # Create figure with 2 subplots
    fig, axes = plt.subplots(2, 1, figsize=(5, 8), gridspec_kw={'height_ratios': [2, 1]})
    
    # Plot replay matrix

    ax1 = axes[0]
    dt_bin = 0.01  # 10 ms in seconds
    n_pos, n_t = replay_matrix.shape

    # imshow with bin EDGES: [0, n_t*dt]
    im = ax1.imshow(
        replay_matrix,
        aspect='auto',
        cmap='hot',
        origin='lower',
        extent=[0, n_t * dt_bin, 0, n_pos],
        vmin=0,          # lower color limit
        vmax=0.3         # upper color limit (example)
    )


    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Position bins")
    plt.colorbar(im, ax=ax1, label="Probability")



    
    # Plot LFP
    ax2 = axes[1]
    if len(lfp_segment) > 0:
        ax2.plot(time_segment - center_time, lfp_segment, 'k', linewidth=0.5)
        ax2.axvspan(interval[0] - center_time, interval[1] - center_time, 
                    alpha=0.3, color='green', label='Replay interval')
        ax2.set_xlabel('Time relative to event center (s)')
        ax2.set_ylabel('LFP (µV)')
        ax2.set_title(f'LFP on channel {lfp_chan}')
        for spine in ["top", "right", "bottom", "left"]:
            ax2.spines[spine].set_visible(False)
        ax2.set_xticks([])
        ax2.set_yticks([])
        ax2.tick_params(
            bottom=False, left=False,
            labelbottom=False, labelleft=False
        )

    save_dir=r"../figure_output"
    save_path = os.path.join(save_dir, f"fig2_B_Example_{animal_name}_{day}_non_replay_event_{random_idx}_{side}.svg")
    plt.savefig(save_path, dpi=300)
    
    plt.tight_layout()
    plt.show()
    
    event_duration = interval[1] - interval[0]
else:
    pass